# USOVA3D Follicle Segmentation: 2D U-Net on Kaggle

This notebook trains a 2D U-Net to segment follicles from axial ultrasound slices. It builds conservative training targets from the overlap of the two expert follicle annotations, evaluates with volume-grouped cross-validation, then trains a final model and exports 3D connected-component labels and measurements for integration into the Flask app.

## Required Kaggle data

The path `/kaggle/input/datasets/mariasiembor/usova-cnn-slices/cnn_slices` is valid, but its `masks` are **ovary masks**, not follicle masks. It can be used by the ovary U-Net notebook, but it cannot train this follicle U-Net by itself. Attach a second Kaggle Dataset containing `Training_Set_1/vol*.vtk` and both `vol*_f_r1.vtk` and `vol*_f_r2.vtk` annotation files, then set `TRAIN_DIR` in the first code cell to that dataset path.

Run this notebook on Kaggle with GPU enabled.

In [ ]:
import glob
import json
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import ndimage
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, Dataset

# Kaggle mounts attached datasets below /kaggle/input with generated folder names.
# Find the folder by its required raw volume and paired follicle annotations.
INPUT_ROOT = Path('/kaggle/input')
required_files = ('vol1.vtk', 'vol1_f_r1.vtk', 'vol1_f_r2.vtk')
candidates = [
    path.parent for path in INPUT_ROOT.rglob('vol1.vtk')
    if all((path.parent / filename).exists() for filename in required_files)
]
if not candidates:
    raise FileNotFoundError(
        'No follicle training dataset found. Attach a Kaggle Dataset containing '
        'vol1.vtk, vol1_f_r1.vtk, and vol1_f_r2.vtk.'
    )
TRAIN_DIR = str(candidates[0])
print(f'Resolved training data: {TRAIN_DIR}')

OUTPUT_DIR = Path('/kaggle/working/follicle_unet')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VOL_IDS = [1, 2, 3, 4, 5, 6, 101, 102, 104, 106, 109, 110, 111, 115, 117, 119]
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 20
LEARNING_RATE = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {DEVICE}; output: {OUTPUT_DIR}')


def read_vtk(path):
    with open(path, 'rb') as file:
        data = file.read()
    header = {}
    pos = 0
    for _ in range(10):
        end = data.index(b'\n', pos)
        line = data[pos:end].decode('latin-1').strip()
        pos = end + 1
        if line.startswith('DIMENSIONS'):
            header['dims'] = tuple(int(x) for x in line.split()[1:])
        elif line.startswith('SPACING'):
            header['spacing'] = tuple(float(x) for x in line.split()[1:])
        elif line.startswith('SCALARS'):
            header['dtype'] = line.split()[2]
        elif line.startswith('LOOKUP_TABLE'):
            break
    dtype = {'unsigned_char': '>u1', 'short': '>i2', 'int': '>i4', 'float': '>f4'}[header['dtype']]
    nx, ny, nz = header['dims']
    return np.frombuffer(data[pos:], dtype=dtype, count=nx * ny * nz).reshape((nz, ny, nx)), header['spacing']


## Load raw volumes and expert follicle targets

Each training volume is loaded with both expert follicle annotations. The target mask is their voxel-wise intersection, which avoids treating disagreements as positive labels.

In [ ]:
def load_all_volumes():
    volumes = {}
    for vid in VOL_IDS:
        raw, spacing = read_vtk(os.path.join(TRAIN_DIR, f'vol{vid}.vtk'))
        expert_a, _ = read_vtk(os.path.join(TRAIN_DIR, f'vol{vid}_f_r1.vtk'))
        expert_b, _ = read_vtk(os.path.join(TRAIN_DIR, f'vol{vid}_f_r2.vtk'))
        # Conservative consensus target: a voxel is positive when both experts marked it.
        target = ((expert_a > 0) & (expert_b > 0)).astype(np.uint8)
        volumes[vid] = {'images': raw.astype(np.uint8), 'masks': target,
                        'spacing': spacing, 'shape': raw.shape}
        print(f'vol{vid}: image {raw.shape}, target voxels {int(target.sum())}')
    return volumes

all_volumes = load_all_volumes()
print(f'Loaded {len(all_volumes)} volumes')


## Slice dataset

Positive slices and a subsample of empty slices are resized for efficient 2D U-Net training. Splits remain grouped by volume to prevent slice-level leakage.

In [ ]:
class SliceDataset(Dataset):
    def __init__(self, volumes, vol_ids, img_size=IMG_SIZE, augment=False):
        self.samples = []
        self.img_size = img_size
        self.augment = augment
        for vid in vol_ids:
            volume = volumes[vid]
            for z, (image, mask) in enumerate(zip(volume['images'], volume['masks'])):
                # Keep positive slices and a limited number of empty slices.
                if mask.any() or z % 4 == 0:
                    self.samples.append((image, mask, vid, z))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image, mask, vid, z = self.samples[index]
        image = cv2.resize(image, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        image = image.astype(np.float32) / 255.0
        if self.augment and np.random.rand() > 0.5:
            image = np.fliplr(image).copy()
            mask = np.fliplr(mask).copy()
        return (torch.from_numpy(image).unsqueeze(0),
                torch.from_numpy(mask.astype(np.float32)).unsqueeze(0), vid, z)


## 2D U-Net model

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.enc1, self.enc2 = DoubleConv(1, base), DoubleConv(base, base * 2)
        self.enc3, self.enc4 = DoubleConv(base * 2, base * 4), DoubleConv(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 8, base * 16)
        self.up4, self.up3 = nn.ConvTranspose2d(base * 16, base * 8, 2, 2), nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.up2, self.up1 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2), nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec4, self.dec3 = DoubleConv(base * 16, base * 8), DoubleConv(base * 8, base * 4)
        self.dec2, self.dec1 = DoubleConv(base * 4, base * 2), DoubleConv(base * 2, base)
        self.out_conv = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1)); e3 = self.enc3(self.pool(e2)); e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1)); d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1)); d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


## Loss (Dice + BCE, standard combo for segmentation) and training loop

In [ ]:
def dice_loss(logits, target, eps=1e-6):
    prediction = torch.sigmoid(logits)
    intersection = (prediction * target).sum(dim=(1, 2, 3))
    denominator = prediction.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return (1 - (2 * intersection + eps) / (denominator + eps)).mean()


def dice_iou(prediction, target):
    prediction = prediction.astype(bool)
    target = target.astype(bool)
    intersection = np.logical_and(prediction, target).sum()
    union = np.logical_or(prediction, target).sum()
    return (2 * intersection / (prediction.sum() + target.sum() + 1e-8),
            intersection / (union + 1e-8))


def train_one_fold(train_ids, valid_ids):
    train_loader = DataLoader(SliceDataset(all_volumes, train_ids, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(SliceDataset(all_volumes, valid_ids), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    model = UNet().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    best_dice, best_state = -1, None
    for epoch in range(EPOCHS):
        model.train()
        for images, masks, _, _ in train_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            logits = model(images)
            loss = dice_loss(logits, masks) + nn.functional.binary_cross_entropy_with_logits(logits, masks)
            loss.backward(); optimizer.step()
        model.eval(); pred, true = [], []
        with torch.no_grad():
            for images, masks, _, _ in valid_loader:
                pred.append((torch.sigmoid(model(images.to(DEVICE))) > 0.5).cpu().numpy())
                true.append(masks.numpy())
        fold_dice, fold_iou = dice_iou(np.concatenate(pred), np.concatenate(true))
        if fold_dice > best_dice:
            best_dice, best_state = fold_dice, {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'epoch {epoch + 1:02d}/{EPOCHS}: validation Dice={fold_dice:.3f}, IoU={fold_iou:.3f}')
    model.load_state_dict(best_state)
    return model, best_dice


## 4-fold grouped cross-validation

Splits the 16 volumes into 4 groups (grouped, not random slice-level
split, to avoid leakage between slices of the same volume). Reconstructs
each held-out volume's predicted mask at ORIGINAL resolution to compute
a fair, directly comparable Dice/IoU against the classical baseline.

In [ ]:
# Grouped folds keep every slice from one volume in the same split.
gkf = GroupKFold(n_splits=4)
fold_results = []
all_ids = np.array(VOL_IDS)
for fold, (train_idx, valid_idx) in enumerate(gkf.split(all_ids, groups=all_ids), start=1):
    train_ids, valid_ids = all_ids[train_idx].tolist(), all_ids[valid_idx].tolist()
    print(f'\nFold {fold}: train={train_ids}, validation={valid_ids}')
    _, fold_dice = train_one_fold(train_ids, valid_ids)
    fold_results.append({'fold': fold, 'validation_volumes': valid_ids, 'dice': float(fold_dice)})

with open(OUTPUT_DIR / 'cross_validation.json', 'w') as file:
    json.dump(fold_results, file, indent=2)
print(f'\nMean grouped validation Dice: {np.mean([r["dice"] for r in fold_results]):.3f}')


## Final comparison against the classical Random Forest baseline

In [ ]:
# Train one final model on all annotated volumes after cross-validation.
final_loader = DataLoader(SliceDataset(all_volumes, VOL_IDS, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
final_model = UNet().to(DEVICE)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=LEARNING_RATE)
for epoch in range(EPOCHS):
    final_model.train(); losses = []
    for images, masks, _, _ in final_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad(); logits = final_model(images)
        loss = dice_loss(logits, masks) + nn.functional.binary_cross_entropy_with_logits(logits, masks)
        loss.backward(); optimizer.step(); losses.append(loss.item())
    print(f'final epoch {epoch + 1:02d}/{EPOCHS}: loss={np.mean(losses):.4f}')

checkpoint_path = OUTPUT_DIR / 'follicle_unet.pt'
torch.save({'model_state_dict': final_model.state_dict(), 'img_size': IMG_SIZE, 'volume_ids': VOL_IDS}, checkpoint_path)


def predict_volume(volume, spacing):
    final_model.eval()
    predicted_slices = []
    with torch.no_grad():
        for image in volume:
            original_shape = image.shape
            resized = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR).astype(np.float32) / 255.0
            tensor = torch.from_numpy(resized).unsqueeze(0).unsqueeze(0).to(DEVICE)
            prediction = (torch.sigmoid(final_model(tensor))[0, 0].cpu().numpy() > 0.5).astype(np.uint8)
            predicted_slices.append(cv2.resize(prediction, (original_shape[1], original_shape[0]), interpolation=cv2.INTER_NEAREST))
    binary = np.stack(predicted_slices)
    labels, count = ndimage.label(binary, structure=ndimage.generate_binary_structure(3, 1))
    min_voxels = (4 / 3 * np.pi * 1.0 ** 3) / np.prod(spacing)
    sizes = ndimage.sum(binary, labels, range(1, count + 1))
    keep = {index + 1 for index, size in enumerate(sizes) if size >= min_voxels}
    labels = np.where(np.isin(labels, list(keep)), labels, 0).astype(np.uint16)
    measurements = []
    voxel_volume = float(np.prod(spacing))
    for label in np.unique(labels[labels > 0]):
        coordinates = np.argwhere(labels == label)
        volume_mm3 = len(coordinates) * voxel_volume
        diameter_mm = 2 * (3 * volume_mm3 / (4 * np.pi)) ** (1 / 3)
        measurements.append({'follicle_id': int(label), 'volume_mm3': volume_mm3,
                             'equivalent_diameter_mm': diameter_mm,
                             'centroid_zyx': coordinates.mean(axis=0).tolist()})
    return labels, measurements

for vid, volume_data in all_volumes.items():
    labels, measurements = predict_volume(volume_data['images'], volume_data['spacing'])
    np.save(OUTPUT_DIR / f'vol{vid}_follicle_labels.npy', labels)
    pd.DataFrame(measurements).to_csv(OUTPUT_DIR / f'vol{vid}_follicle_measurements.csv', index=False)

print(f'Saved model and {len(all_volumes)} 3D prediction volumes to {OUTPUT_DIR}')
